# Welcome to Colab!

In [2]:
# =========================
# Install dependencies
# =========================
!pip install pandas numpy pyyaml -q

# =========================
# Imports
# =========================
import pandas as pd
import numpy as np
import yaml
import json
import time
import logging
import os

# =========================
# 👉 CREATE config.yaml
# =========================
config_data = {
    "seed": 42,
    "window": 5,
    "version": "v1"
}

with open("config.yaml", "w") as f:
    yaml.dump(config_data, f)

# =========================
# 👉 SET YOUR DATASET PATH HERE
# =========================
DATASET_PATH = "/content/data.csv - Sheet1.pdf"
# 🔴 CHANGE THIS:
# Example:
# DATASET_PATH = "/content/drive/MyDrive/data/data.csv"

# =========================
# Setup logger
# =========================
logging.basicConfig(
    filename="run.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# =========================
# Start job
# =========================
start_time = time.time()

try:
    logging.info("Job started")

    # -------- Load config --------
    if not os.path.exists("config.yaml"):
        raise Exception("Config file not found")

    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

    for key in ["seed", "window", "version"]:
        if key not in config:
            raise Exception(f"Missing config key: {key}")

    seed = config["seed"]
    window = config["window"]
    version = config["version"]

    np.random.seed(seed)

    logging.info(f"Config loaded: {config}")

    # -------- Load dataset --------
    if not os.path.exists(DATASET_PATH):
        raise Exception("Input file not found")

    df = pd.read_csv(DATASET_PATH)

    if df.empty:
        raise Exception("CSV is empty")

    if "close" not in df.columns:
        raise Exception("Missing 'close' column")

    logging.info(f"Rows loaded: {len(df)}")

    # -------- Rolling mean --------
    df["rolling_mean"] = df["close"].rolling(window=window).mean()

    # -------- Signal --------
    df["signal"] = np.where(df["close"] > df["rolling_mean"], 1, 0)

    # Remove NaN rows
    valid_df = df.dropna()

    signal_rate = valid_df["signal"].mean()
    rows_processed = len(df)

    latency_ms = int((time.time() - start_time) * 1000)

    metrics = {
        "version": version,
        "rows_processed": rows_processed,
        "metric": "signal_rate",
        "value": round(float(signal_rate), 4),
        "latency_ms": latency_ms,
        "seed": seed,
        "status": "success"
    }

    # -------- Write metrics --------
    with open("metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    logging.info(f"Metrics: {metrics}")
    logging.info("Job completed successfully")

    print("✅ SUCCESS OUTPUT:")
    print(json.dumps(metrics, indent=2))

except Exception as e:
    error_metrics = {
        "version": "v1",
        "status": "error",
        "error_message": str(e)
    }

    with open("metrics.json", "w") as f:
        json.dump(error_metrics, f, indent=2)

    logging.error(str(e))

    print("❌ ERROR OUTPUT:")
    print(json.dumps(error_metrics, indent=2))

ERROR:root:'utf-8' codec can't decode byte 0xe2 in position 11: invalid continuation byte


❌ ERROR OUTPUT:
{
  "version": "v1",
  "status": "error",
  "error_message": "'utf-8' codec can't decode byte 0xe2 in position 11: invalid continuation byte"
}


In [ ]:
step-1:Create run.py

In [ ]:
import argparse
import yaml
import pandas as pd
import numpy as np
import json
import time
import logging
import sys
import os

def setup_logger(log_file):
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )

def write_metrics(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--config", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--log-file", required=True)
    args = parser.parse_args()

    setup_logger(args.log_file)
    start_time = time.time()

    try:
        logging.info("Job started")

        # CONFIG
        if not os.path.exists(args.config):
            raise Exception("Config file not found")

        with open(args.config) as f:
            config = yaml.safe_load(f)

        for k in ["seed", "window", "version"]:
            if k not in config:
                raise Exception(f"Missing config key: {k}")

        seed = config["seed"]
        window = config["window"]
        version = config["version"]

        np.random.seed(seed)

        logging.info(f"Config validated: {config}")

        # DATA
        if not os.path.exists(args.input):
            raise Exception("Input file not found")

        df = pd.read_csv(args.input)

        if df.empty:
            raise Exception("CSV is empty")

        if "close" not in df.columns:
            raise Exception("Missing 'close' column")

        logging.info(f"Rows loaded: {len(df)}")

        # PROCESS
        logging.info("Computing rolling mean")
        df["rolling_mean"] = df["close"].rolling(window=window).mean()

        logging.info("Generating signal")
        df["signal"] = np.where(df["close"] > df["rolling_mean"], 1, 0)

        valid_df = df.dropna()

        signal_rate = valid_df["signal"].mean()
        rows_processed = len(df)

        latency_ms = int((time.time() - start_time) * 1000)

        metrics = {
            "version": version,
            "rows_processed": rows_processed,
            "metric": "signal_rate",
            "value": round(float(signal_rate), 4),
            "latency_ms": latency_ms,
            "seed": seed,
            "status": "success"
        }

        write_metrics(args.output, metrics)

        logging.info(f"Metrics: {metrics}")
        logging.info("Job completed successfully")

        print(json.dumps(metrics, indent=2))
        sys.exit(0)

    except Exception as e:
        error_metrics = {
            "version": "v1",
            "status": "error",
            "error_message": str(e)
        }

        write_metrics(args.output, error_metrics)

        logging.error(str(e))
        print(json.dumps(error_metrics, indent=2))

        sys.exit(1)

if __name__ == "__main__":
    main()

In [ ]:
step-2:Create config.yaml

In [ ]:
seed: 42
window: 5
version: "v1"

In [ ]:
STEP 4: requirements.txt

In [ ]:
pandas
numpy
pyyaml

In [ ]:
STEP 5: Dockerfile

In [ ]:
FROM python:3.9-slim

WORKDIR /app

COPY . /app

RUN pip install --no-cache-dir -r requirements.txt

CMD ["python", "run.py",
     "--input", "data.csv",
     "--config", "config.yaml",
     "--output", "metrics.json",
     "--log-file", "run.log"]

In [ ]:
STEP 6: Run locally (VERY IMPORTANT)

In [ ]:
python run.py --input data.csv --config config.yaml --output metrics.json --log-file run.log

In [ ]:
STEP 7: Docker test (MUST PASS)

In [ ]:
docker build -t mlops-task .
docker run --rm mlops-task

In [ ]:
STEP 8: README.md

In [ ]:
# MLOps Task

## Run locally
python run.py --input data.csv --config config.yaml --output metrics.json --log-file run.log

## Docker
docker build -t mlops-task .
docker run --rm mlops-task

## Output
metrics.json and run.log